# Simple MS Simulator — Phase 0: ICEBERG gate (Colab)

`specs_simple_ms_simulator/roadmap.md` Phase 0. **Generated by `build/mk_simple_sim_gate.py` —
do not hand-edit.** Runtime: Colab **GPU (L4 or A100) + High-RAM**, roughly 1–1.5 h, most of it
the one-off ms-pred install and building the offline wheelhouse.

| gate | question | how it is answered here |
|---|---|---|
| G1 | licence / provenance | MassSpecGym-trained checkpoints from the ms-pred README, pinned commit, SHA1 recorded |
| G2 | adduct coverage | per-adduct fidelity + discrimination on real timsTOF spectra, 10 test adducts |
| G2b | isobaric coverage (decides coverage) | truth ranked among ±8.5 ppm pool candidates — the tier-2 task itself |
| G3 | CE conditioning | eV vs NCE input compared; prediction at 10 vs 60 eV compared |
| G4 | throughput | mol/s on ~1,000 COCONUT structures across 150–1,200 Da, GPU and CPU |
| G5 | offline install | wheelhouse + checkpoints written to Drive; **proved on Kaggle** by `casmi_simple_sim_gate_kaggle.ipynb` |
| G6 | output sanity | after the worker's precursor + 2 Da cut: non-empty, finite; per-adduct report of what the cut removed |

ICEBERG runs in its **own** environment (torch 2.6, DGL, RDKit 2025.03) as a subprocess
(`iceberg_worker.py`). The notebook's interpreter never imports it, so the metric's RDKit
2026.03.3 elsewhere in the project is never disturbed.

**A failed gate means stop and re-plan** (mission.md §5) — there is no fallback simulator.

**Re-run mode (revision 2, 2026-09-27).** The first run passed G1, G3, G4 and G5; G6 was
mis-specified and G2 was too lenient (`specs_simple_ms_simulator/gate_report.md`). With
`RERUN = True` the sweep is cut to the best checkpoint × {eV, NCE} × QTOF (G3 needs both CE units)
and G4/G5 are carried from the previous `gate_report.json` instead of re-measured, which skips the
wheelhouse build. Set `RERUN = False` for a full run from scratch.
The last cell writes `_simlib/gate/gate_report.json`; paste its printout back into the chat.

In [ ]:
import os, sys, json, time, glob, shutil, hashlib, pickle, subprocess, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/casmi_2026")
COMP = ROOT / "enveda-CASMI26-molecule-id-mass-spectra"
TRAIN, TEST = COMP / "train.parquet", COMP / "test.parquet"
COCO = ROOT / "coconut_dataset"
GATE = ROOT / "_simlib" / "gate"
UPLOAD = GATE / "kaggle_upload"                # becomes the Kaggle dataset casmi26-iceberg-offline
WORK = Path("/content/simgate")
for p in (GATE, WORK):
    p.mkdir(parents=True, exist_ok=True)

MSPRED_DIR = Path("/content/ms-pred")
MSPRED_COMMIT = "708148c2a8eb0c32120644436fefd2fe90cd2214"     # pinned: the commit the spec was written against
# MassSpecGym-trained ICEBERG 2.1 checkpoints (README of the pinned commit). NIST weights are forbidden.
CKPT_URLS = {
    "msg_all": "https://www.dropbox.com/scl/fo/kwm35ih8tlfnshfrcq8ot/AOeS4M0_v9MhqeEZys9sCRQ?rlkey=f1n6pbzx94g1k2el2wcbmee61&dl=1",
    "msg_simulation": "https://www.dropbox.com/scl/fo/mcj0ngdvuj2xkhr983frb/ABGLrS8ZCeyYUWhzRfxPtRg?rlkey=rccvo52cehh75ma1yy8jji46t&dl=1",
}
KAGGLE_USER = "YOUR_KAGGLE_USERNAME"           # only used in the dataset-metadata.json that is written

TEST_ADDUCTS = ["[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",
                "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-"]
PRIORITY_LIBS = ("enveda-np-examples", "enveda-180")   # same timsTOF + pipeline as the test set
N_PER_ADDUCT = 40
SEED = 0

# gate thresholds -- stated before the numbers are seen
COVER_MIN_GAP = 0.05          # median sim(real, own prediction) - median sim(real, other molecule's prediction)
COVER_MIN_TOP1_X_CHANCE = 3.0 # own prediction ranked first at >= 3x the chance rate within the adduct group
SANITY_MIN_OK = 0.98          # G6: share of predictions that are non-empty, finite and <= precursor + 2 Da
CE_EFFECTIVE_MAX_SIM = 0.90   # G3: median sim(pred@10 eV, pred@60 eV) below this => CE actually conditions
# G2b (revision 2) -- stated before the numbers are seen. Coverage is decided here, not by G2.
ISO_PPM = 8.5                 # decoys: pool structures within +-8.5 ppm of the truth's exact neutral mass
ISO_MAX_DECOYS = 20           # per query, random subset (window median is ~40-60)
ISO_MIN_QUERIES = 15          # an adduct needs this many queries with >= 1 decoy to be judged
ISO_MIN_LIFT = 2.0            # top-1 rate >= 2x the chance top-1 rate ...
ISO_MAX_P = 0.01              # ... and one-sided p < 0.01 against chance (exact Poisson-binomial)
PREC_MARGIN = 2.0             # worker drops predicted peaks above precursor + 2 Da (same cut as the test spectra)
# where include_unshifted_mz puts the unshifted root, relative to the precursor (finding 1 of the first run)
EXPECTED_UNSHIFTED_OFF = {"[M-H2O+H]+": 17.003, "[M-2H2O+H]+": 35.013, "[M-H2O-H]-": 19.018}

RERUN = True                  # revision 2 re-run: reduced sweep, G4/G5 carried from the previous report
RERUN_CKPT, RERUN_INSTR = "msg_all", "QTOF"   # the first run's best variant was msg_all_ev_QTOF

PREV = json.loads((GATE / "gate_report.json").read_text()) if (GATE / "gate_report.json").exists() else {}
if RERUN:
    assert "G4" in PREV and "G5" in PREV, "RERUN needs the previous gate_report.json with G4/G5 -- set RERUN = False"
    print(f"RERUN: carrying G4/G5 from the run started {PREV['started']}")
GATE_RESULT = {"mspred_commit": MSPRED_COMMIT, "started": time.strftime("%Y-%m-%d %H:%M"),
               "revision": 2, "rerun": RERUN}

def save_gate():
    (GATE / "gate_report.json").write_text(json.dumps(GATE_RESULT, indent=1, default=str))

def sh(cmd, timeout=None, env=None):
    print("$", cmd if len(cmd) < 300 else cmd[:300] + " ...", flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout, env=env)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode:
        print(r.stderr[-4000:])
        raise RuntimeError(f"command failed ({r.returncode})")
    return r.stdout

print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))
print(f"vCPU {os.cpu_count()}")

## Install ms-pred into an isolated environment (~10 min, once per session)

In [ ]:
# Colab sets PYTHONPATH to its own site-packages; the venv must not see them.
VENV_ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}
VENV_ENV["DGLBACKEND"] = "pytorch"
IPY = str(MSPRED_DIR / ".venv" / "bin" / "python")

if not Path(IPY).exists():
    sh("pip -q install uv")
    if not MSPRED_DIR.exists():
        sh(f"git clone -q https://github.com/coleygroup/ms-pred.git {MSPRED_DIR}")
    sh(f"cd {MSPRED_DIR} && git checkout -q {MSPRED_COMMIT}")
    sh(f"cd {MSPRED_DIR} && uv sync --extra cu124 --python 3.12", timeout=3600, env=VENV_ENV)

raw_info = sh(f'{IPY} -c "import sys, torch, dgl, rdkit, ms_pred; '
              f'print(sys.version.split()[0], torch.__version__, torch.cuda.is_available(), dgl.__version__, rdkit.__version__)"',
              env=VENV_ENV)
info = raw_info.strip().splitlines()[-1].split()
GATE_RESULT["env"] = dict(python=info[0], torch=info[1], cuda=info[2], dgl=info[3], rdkit=info[4])
assert info[2] == "True", "torch in the ms-pred venv cannot see the GPU"
print(GATE_RESULT["env"])

## Worker script (embedded from `build/iceberg_worker.py`)

In [ ]:
WORKER_SRC = '"""ICEBERG forward-simulation worker for specs_simple_ms_simulator.\n\nRuns INSIDE the isolated ms-pred environment (its own torch 2.6 / DGL / rdkit 2025.03), never in\nthe notebook\'s main interpreter -- that keeps RDKit 2026.03.3 (the metric\'s version) untouched.\nThe notebook talks to it through files only:\n\n    request tsv      : key, smiles, adduct (CASMI spelling), precursor_mz, ce, instrument\n                       (TSV, not parquet: the ms-pred environment ships no pyarrow)\n    response npz     : off (n+1,) int64, mz / it float32 flat, ok bool, err str, timing json,\n                       above_n int32 / above_share float32 / above_off float32 (per row, see below)\n\nPeaks are the top `--sparse-k` by intensity, sorted by m/z, base peak = 1.0. m/z values are the\ncharged fragment masses (ICEBERG applies the adduct shift itself).\n\nPeaks above precursor_mz + `--prec-margin` (2 Da) are dropped BEFORE the top-k, the same cleaning the\norganisers applied to the test spectra. They exist because the MassSpecGym inten checkpoints have\n`include_unshifted_mz=True`: every fragment is also emitted without the adduct shift, and for\nwater-loss adducts the unshifted root lands 17-35 Da above the precursor. What was dropped is\nrecorded per row: above_n = peak count, above_share = share of raw intensity, above_off = the\nlargest dropped m/z minus the precursor (NaN when nothing was dropped). A row left with no peaks\ngets err "empty_after_prec_cut".\n\nReused unchanged by Phase 0 (gate), Phase 2 (precompute) and Phase 4 (Kaggle on-the-fly).\n\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --inp req.tsv --out resp.npz\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --hparams-only --out hp.json\n"""\nimport argparse\nimport json\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom rdkit import Chem, RDLogger\n\nimport importlib.abc  # noqa: E402\nimport importlib.machinery  # noqa: E402\nimport os  # noqa: E402\nimport sys  # noqa: E402\nimport types  # noqa: E402\n\nSTUBBED = []\n\n\nclass _Missing:\n    """Stands in for a name from an ms_pred module the package does not ship; fails only if actually used."""\n\n    def __init__(self, name):\n        self._name = name\n\n    def __call__(self, *a, **k):\n        raise RuntimeError(f"{self._name} is not shipped with this ms_pred copy")\n\n    def __getattr__(self, attr):\n        raise RuntimeError(f"{self._name}.{attr} is not shipped with this ms_pred copy")\n\n\nclass _StubMissingMsPred(importlib.abc.MetaPathFinder, importlib.abc.Loader):\n    """The casmi26-iceberg package ships a trimmed ms_pred (inference modules only) whose package __init__s still\n    import training-side modules (e.g. nn_utils.tune_utils). Last on sys.meta_path, so it only answers for an\n    ms_pred submodule no real finder can find, with an empty module."""\n\n    def find_spec(self, name, path=None, target=None):\n        return importlib.machinery.ModuleSpec(name, self) if name.startswith("ms_pred.") else None\n\n    def create_module(self, spec):\n        m = types.ModuleType(spec.name)\n        m.__all__ = []\n\n        def __getattr__(attr, _n=spec.name):\n            if attr.startswith("__"):\n                raise AttributeError(attr)\n            return _Missing(f"{_n}.{attr}")\n        m.__getattr__ = __getattr__\n        return m\n\n    def exec_module(self, module):\n        STUBBED.append(module.__name__)\n\n\nclass _Anything:\n    """Permissive placeholder served by an absent third-party module (import-time use only; ice_runner.py\'s)."""\n\n    def __init__(self, *a, **k):\n        pass\n\n    def __call__(self, *a, **k):\n        return _Anything()\n\n    def __getattr__(self, k):\n        if k.startswith("__"):\n            raise AttributeError(k)\n        return _Anything()\n\n    def __getitem__(self, k):\n        return _Anything()\n\n    def __setitem__(self, k, v):\n        pass\n\n    def __iter__(self):\n        return iter(())\n\n    def __bool__(self):\n        return False\n\n\nclass _StubModule(types.ModuleType):\n    def __getattr__(self, k):\n        if k.startswith("__"):\n            raise AttributeError(k)\n        v = type(k, (_Anything,), {}) if k[:1].isupper() else _Anything()\n        setattr(self, k, v)\n        return v\n\n\nclass _StubThirdParty(importlib.abc.MetaPathFinder, importlib.abc.Loader):\n    """Serves stub modules (and any submodule) for third-party roots that ms_pred imports at module level but never\n    uses on the predict_mol path, when the image lacks them (multiprocess, h5py, pygmtools on the cp313 image)."""\n\n    def __init__(self, roots):\n        self.roots = set(roots)\n\n    def find_spec(self, name, path=None, target=None):\n        if name.split(".")[0] not in self.roots:\n            return None\n        return importlib.machinery.ModuleSpec(name, self, is_package=True)\n\n    def create_module(self, spec):\n        m = _StubModule(spec.name)\n        m.__path__ = []\n        return m\n\n    def exec_module(self, module):\n        pass\n\n\n# the casmi26-iceberg ice_runner.py list: imported at ms_pred module level, unused by ICEBERG inference\nOPTIONAL = ("pygmtools", "h5py", "multiprocess", "platformdirs")\n\nif os.environ.get("CASMI_ICE_PKG") == "1":\n    sys.meta_path.append(_StubMissingMsPred())\n    _absent = []\n    for _m in OPTIONAL:\n        try:\n            importlib.import_module(_m)\n        except BaseException:\n            for _k in [k for k in sys.modules if k.split(".")[0] == _m]:\n                del sys.modules[_k]\n            _absent.append(_m)\n    if _absent:\n        sys.meta_path.insert(0, _StubThirdParty(_absent))\n        print(f"[worker] absent third-party modules, stubbed: {_absent}", file=sys.stderr, flush=True)\n\nimport ms_pred.common as common  # noqa: E402\nimport ms_pred.common.chem_utils as _chem_utils  # noqa: E402\nfrom ms_pred.iceberg import gen_model, inten_model, joint_model  # noqa: E402\n\nif STUBBED:\n    print(f"[worker] ms_pred modules not shipped, stubbed: {STUBBED}", file=sys.stderr, flush=True)\n\n# predict_mol\'s InChI round trip writes a constant tautomer-transform file under platformdirs\' user_cache_dir:\n# point it at a writable temp dir (also needed when platformdirs itself is stubbed)\nimport tempfile  # noqa: E402\n_CACHE = os.path.join(tempfile.gettempdir(), "ice_ms_pred_cache")\n_chem_utils.user_cache_dir = lambda name, *a, **k: os.path.join(_CACHE, name)\n\nimport rdkit  # noqa: E402\nprint(f"[worker] rdkit {rdkit.__version__} at {os.path.dirname(rdkit.__file__)} | torch {torch.__version__}",\n      file=sys.stderr, flush=True)\nif os.environ.get("CASMI_ICE_PKG") == "1" and not rdkit.__version__.startswith("2025.03"):\n    raise RuntimeError(f"rdkit {rdkit.__version__}: the casmi26-iceberg checkpoints need 2025.03 "\n                       "(2026.03 kekulizes differently) -- the ice site is not first on PYTHONPATH")\n\nRDLogger.DisableLog("rdApp.*")\n\n# CASMI adduct spelling -> ms-pred spelling (only where they differ)\nADDUCT_MAP = {"[M-2H2O+H]+": "[M+H-2H2O]+", "[M+CH2O2-H]-": "[M+CHO2]-"}\n\n\ndef get_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--gen", required=True)\n    p.add_argument("--inten", required=True)\n    p.add_argument("--inp")\n    p.add_argument("--out", required=True)\n    p.add_argument("--device", default="cuda")\n    p.add_argument("--batch-size", type=int, default=64)\n    p.add_argument("--max-nodes", type=int, default=100)\n    p.add_argument("--sparse-k", type=int, default=100)\n    p.add_argument("--threshold", type=float, default=0.0)\n    p.add_argument("--prec-margin", type=float, default=2.0)\n    p.add_argument("--hparams-only", action="store_true")\n    return p.parse_args()\n\n\ndef _jsonable(d):\n    out = {}\n    for k, v in dict(d).items():\n        try:\n            json.dumps(v)\n            out[k] = v\n        except TypeError:\n            out[k] = str(v)\n    return out\n\n\n# PyTorch 2.6 defaults weights_only=True, which breaks legacy Lightning checkpoints containing PosixPath/hparams.\ntry:\n    from pathlib import PosixPath, WindowsPath\n    torch.serialization.add_safe_globals([PosixPath, WindowsPath])\nexcept Exception:\n    pass\n\n_orig_torch_load = torch.load\ndef _safe_torch_load(*load_args, **load_kwargs):\n    if "weights_only" not in load_kwargs:\n        load_kwargs["weights_only"] = False\n    return _orig_torch_load(*load_args, **load_kwargs)\ntorch.load = _safe_torch_load\n\n\nHP_KEYS = ("hyper_parameters", "hparams", "hp", "config", "kwargs", "args")\nSD_KEYS = ("state_dict", "model_state_dict", "model", "weights")\n\n\ndef load_model(cls, path):\n    """A Lightning checkpoint (.ckpt), or a plain torch file (.pt, e.g. the casmi26-iceberg package): a pickled\n    module, or a dict holding hyper-parameters + state_dict."""\n    if str(path).endswith(".pt"):   # the casmi26-iceberg layout: {hyper_parameters, state_dict}, as ice_runner.py loads it\n        obj = torch.load(path, map_location="cpu")\n        if isinstance(obj, dict) and "hyper_parameters" in obj and "state_dict" in obj:\n            m = cls(**obj["hyper_parameters"])\n            m.load_state_dict(obj["state_dict"], strict=True)\n            return m\n    try:\n        try:\n            return cls.load_from_checkpoint(path, map_location="cpu", weights_only=False)\n        except TypeError:\n            return cls.load_from_checkpoint(path, map_location="cpu")\n    except Exception as e:\n        lightning_err = e\n    obj = torch.load(path, map_location="cpu")\n    if isinstance(obj, torch.nn.Module):\n        return obj\n    if isinstance(obj, dict):\n        hp = next((obj[k] for k in HP_KEYS if k in obj), None)\n        sd = next((obj[k] for k in SD_KEYS if k in obj), None)\n        if isinstance(sd, torch.nn.Module):\n            return sd\n        if hp is not None and sd is not None:\n            m = cls(**dict(vars(hp) if hasattr(hp, "__dict__") and not isinstance(hp, dict) else hp))\n            m.load_state_dict(sd)\n            return m\n    raise RuntimeError(f"cannot load {path} as {cls.__name__}: Lightning said {lightning_err!r}; file holds "\n                       f"{sorted(obj)[:30] if isinstance(obj, dict) else type(obj)}")\n\n\ndef load(args):\n    return load_model(gen_model.FragGNN, args.gen), load_model(inten_model.IntenGNN, args.inten)\n\n\ndef to_peaks(s, k, prec, margin):\n    """(n, 2) array [mz, inten] -> (mz, it, above_n, above_share, above_off).\n\n    Drops peaks above prec + margin, then keeps the top-k by intensity, sorted by m/z, base = 1.\n    """\n    s = np.asarray(s, dtype=np.float64).reshape(-1, 2)\n    s = s[np.isfinite(s).all(axis=1) & (s[:, 1] > 0)]\n    above = s[:, 0] > prec + margin\n    above_n = int(above.sum())\n    above_share = float(s[above, 1].sum() / s[:, 1].sum()) if len(s) else 0.0\n    above_off = float(s[above, 0].max() - prec) if above_n else np.nan\n    s = s[~above]\n    if len(s) == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32), above_n, above_share, above_off\n    s = s[np.argsort(s[:, 1])[::-1][:k]]\n    s = s[np.argsort(s[:, 0])]\n    return (s[:, 0].astype(np.float32), (s[:, 1] / s[:, 1].max()).astype(np.float32),\n            above_n, above_share, above_off)\n\n\ndef predict_batch(model, rows, args):\n    return model.predict_mol(\n        [r["smiles"] for r in rows],\n        collision_eng=[float(r["ce"]) for r in rows],\n        precursor_mz=[float(r["precursor_mz"]) for r in rows],\n        adduct=[r["adduct"] for r in rows],\n        instrument=[r["instrument"] for r in rows],\n        threshold=args.threshold,\n        device=args.device,\n        max_nodes=args.max_nodes,\n        adduct_shift=False,\n        canonical_root_smi=True,\n        binned_out=False,\n    )["spec"]\n\n\ndef main():\n    args = get_args()\n    t0 = time.time()\n    gen, inten = load(args)\n\n    if args.hparams_only:\n        with open(args.out, "w") as f:\n            json.dump({"gen": _jsonable(gen.hparams), "inten": _jsonable(inten.hparams),\n                       "ion2onehot_pos": common.ion2onehot_pos,\n                       "instrument2onehot_pos": common.instrument2onehot_pos}, f, indent=1)\n        return\n\n    model = joint_model.JointModel(gen_model_obj=gen, inten_model_obj=inten).to(args.device)\n    model.eval()\n    model.freeze()\n    t_load = time.time() - t0\n\n    df = pd.read_csv(args.inp, sep="\t")\n    n = len(df)\n    peaks = [None] * n\n    err = [""] * n\n    above_n = np.zeros(n, np.int32)\n    above_share = np.zeros(n, np.float32)\n    above_off = np.full(n, np.nan, np.float32)\n\n    # canonicalise exactly as predict_smis.py does; reject what ms-pred cannot take\n    rows = []\n    for i, r in enumerate(df.to_dict("records")):\n        m = Chem.MolFromSmiles(r["smiles"]) if isinstance(r["smiles"], str) else None\n        ad = ADDUCT_MAP.get(r["adduct"], r["adduct"])\n        if m is None:\n            err[i] = "rdkit_parse"\n        elif ad not in common.ion2onehot_pos:\n            err[i] = f"adduct_unsupported:{r[\'adduct\']}"\n        elif r["instrument"] not in common.instrument2onehot_pos:\n            err[i] = f"instrument_unsupported:{r[\'instrument\']}"\n        elif not np.isfinite(r["ce"]):\n            err[i] = "ce_nan"\n        else:\n            rows.append(dict(r, i=i, smiles=Chem.MolToSmiles(m), adduct=ad))\n\n    t1 = time.time()\n    with torch.no_grad():\n        for b in range(0, len(rows), args.batch_size):\n            batch = rows[b:b + args.batch_size]\n            try:\n                specs = predict_batch(model, batch, args)\n            except Exception:\n                # isolate the failing molecule(s); keep the rest of the batch\n                specs = []\n                for r in batch:\n                    try:\n                        specs.append(predict_batch(model, [r], args)[0])\n                    except Exception as e:  # noqa: BLE001\n                        specs.append(None)\n                        err[r["i"]] = f"predict:{type(e).__name__}:{str(e)[:120]}"\n            for r, s in zip(batch, specs):\n                if s is not None:\n                    i = r["i"]\n                    mz_, it_, above_n[i], above_share[i], above_off[i] = to_peaks(\n                        s.detach().cpu().numpy(), args.sparse_k, float(r["precursor_mz"]), args.prec_margin)\n                    peaks[i] = (mz_, it_)\n    if args.device.startswith("cuda"):\n        torch.cuda.synchronize()\n    t_pred = time.time() - t1\n\n    empty = (np.empty(0, np.float32), np.empty(0, np.float32))\n    ok = np.array([p is not None and len(p[0]) > 0 for p in peaks])\n    for i in range(n):\n        if peaks[i] is not None and len(peaks[i][0]) == 0 and not err[i]:\n            err[i] = "empty_after_prec_cut" if above_n[i] else "empty_spectrum"\n    peaks = [p if p is not None else empty for p in peaks]\n    lens = np.array([len(p[0]) for p in peaks], dtype=np.int64)\n    off = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    mz = np.concatenate([p[0] for p in peaks]) if n else np.empty(0, np.float32)\n    it = np.concatenate([p[1] for p in peaks]) if n else np.empty(0, np.float32)\n    timing = dict(n=n, n_predicted=int(ok.sum()), t_load=t_load, t_pred=t_pred,\n                  mol_per_s=float(len(rows) / t_pred) if t_pred > 0 else None,\n                  device=args.device, batch_size=args.batch_size, prec_margin=args.prec_margin,\n                  gpu=torch.cuda.get_device_name(0) if args.device.startswith("cuda") else "cpu")\n    np.savez(args.out, off=off, mz=mz, it=it, ok=ok, err=np.array(err, dtype="U200"),\n             above_n=above_n, above_share=above_share, above_off=above_off, timing=json.dumps(timing))\n    print(json.dumps(timing))\n\n\nif __name__ == "__main__":\n    main()\n'
WORKER = WORK / "iceberg_worker.py"
WORKER.write_text(WORKER_SRC)
print(f"wrote {WORKER} ({len(WORKER_SRC):,} chars)")

## G1 — checkpoints: download, fingerprint, inspect

In [ ]:
def sha1(path, chunk=1 << 20):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def pick_ckpts(d):
    found = sorted(d.rglob("*.ckpt"))
    gen = [p for p in found if p.parent.name == "gen"]
    inten = [p for p in found if p.parent.name.startswith("inten")]
    inten = sorted(inten, key=lambda p: (p.parent.name != "inten_contr", str(p)))  # prefer the contrastive-finetuned head
    return found, gen, inten

CK = {}
for name, url in CKPT_URLS.items():
    d = WORK / "ckpt" / name
    if not any(d.rglob("*.ckpt")):
        d.mkdir(parents=True, exist_ok=True)
        z = d / "download.zip"
        sh(f'wget -q -O "{z}" "{url}"', timeout=1800)
        with zipfile.ZipFile(z) as zf:          # Dropbox folder zips carry a '/' entry that trips unzip
            zf.extractall(d)
        z.unlink()
    found, gen, inten = pick_ckpts(d)
    print(f"\n{name}: {len(found)} .ckpt files")
    for p in found:
        print(f"   {p.relative_to(d)}  ({p.stat().st_size/1e6:.1f} MB)")
    if not gen or not inten:
        raise RuntimeError(f"G1 FAIL: {name} has no gen/ and inten*/ checkpoint pair")
    CK[name] = dict(gen=str(gen[0]), inten=str(inten[0]))

G1 = {}
for name, c in CK.items():
    hp_path = WORK / f"hparams_{name}.json"
    sh(f'{IPY} {WORKER} --gen "{c["gen"]}" --inten "{c["inten"]}" --hparams-only --out {hp_path}', env=VENV_ENV)
    hp = json.loads(hp_path.read_text())
    keep = ("embed_adduct", "embed_collision", "embed_instrument", "encode_forms", "include_unshifted_mz",
            "max_broken", "hidden_size", "dataset_name")
    G1[name] = dict(url=CKPT_URLS[name],
                    gen=Path(c["gen"]).relative_to(WORK / "ckpt" / name).as_posix(),
                    inten=Path(c["inten"]).relative_to(WORK / "ckpt" / name).as_posix(),
                    gen_sha1=sha1(c["gen"]), inten_sha1=sha1(c["inten"]),
                    gen_hparams={k: hp["gen"].get(k) for k in keep if k in hp["gen"]},
                    inten_hparams={k: hp["inten"].get(k) for k in keep if k in hp["inten"]})
    ions = hp["ion2onehot_pos"]
    print(f"\n{name}\n  gen   {G1[name]['gen_sha1']}  {G1[name]['gen_hparams']}"
          f"\n  inten {G1[name]['inten_sha1']}  {G1[name]['inten_hparams']}")
unsupported = [a for a in TEST_ADDUCTS
               if {"[M-2H2O+H]+": "[M+H-2H2O]+", "[M+CH2O2-H]-": "[M+CHO2]-"}.get(a, a) not in ions]
print(f"\nadducts the code refuses outright: {unsupported or 'none'}")
print("NOTE: the one-hot table accepting an adduct does NOT mean the MassSpecGym checkpoint was "
      "trained on it -- G2 measures that.")
GATE_RESULT["G1"] = dict(checkpoints=G1, licence="MIT (ms-pred) / MIT (MassSpecGym)",
                         adducts_refused_by_code=unsupported)
save_gate()

## Spectrum kernels (embedded verbatim from phase1 code cell 2)

In [ ]:
import numpy as np
from numba import njit, prange

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx: mx = it[i]
    if mx <= 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx
    c = 0
    for i in range(n):
        if it[i] >= thr: c += 1
    idx = np.empty(c, np.int64)
    j = 0
    for i in range(n):
        if it[i] >= thr:
            idx[j] = i; j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c): v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk): k2[i] = idx[o[i]]
        k2.sort()
        idx = k2
        c = topk
    om = np.empty(c, np.float32)
    oi = np.empty(c, np.float32)
    s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]
        v = it[idx[i]] ** power
        oi[i] = v
        s += v
    if s > 0:
        for i in range(c): oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0: S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S
            s2 = 0.0
            for i in range(c):
                oi[i] = oi[i] ** w
                s2 += oi[i]
            if s2 > 0:
                for i in range(c): oi[i] /= s2
    return om, oi

@njit(cache=True, fastmath=True)
def entropy_sim(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        d = qmz[i] - cmz[j]
        if d < -tol:
            buf[b] = qp[i]; i += 1; b += 1
        elif d > tol:
            buf[b] = cp[j]; j += 1; b += 1
        else:
            buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n: buf[b] = qp[i]; i += 1; b += 1
    while j < m: buf[b] = cp[j]; j += 1; b += 1
    for x in range(b): tot += buf[x]
    if tot <= 0: return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0: SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)

@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):
    """Evaluates max of direct and mass-shifted spectral entropy similarity."""
    a = entropy_sim(qmz, qp, cmz, cp, tol)
    if shift > -0.001 and shift < 0.001:
        return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)):
        sm[i] = cmz[i] + shift
    b = entropy_sim(qmz, qp, sm, cp, tol)
    return a if a > b else b

@njit(cache=True, fastmath=True, parallel=True)
def search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim(qmz, qp, cm, cp, tol)
    return out

@njit(cache=True, fastmath=True, parallel=True)
def search_shift(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])
    return out

In [ ]:
FLOOR, TOPK, POWER, ENTW, TOL = 0.002, 64, 1.0, True, 0.01     # phase1 CFG values

def prep(mz, it, prec):
    mz = np.asarray(mz, np.float32); it = np.asarray(it, np.float32)
    keep = mz <= prec + 2.0
    return _clean(mz[keep], it[keep], FLOOR, TOPK, POWER, ENTW)

def esim(a, b):
    if len(a[0]) == 0 or len(b[0]) == 0:
        return 0.0
    return float(entropy_sim(a[0], a[1], b[0], b[1], TOL))

## Fidelity sample — real timsTOF spectra for the 10 test adducts

One spectrum per structure, single known collision energy (eV), ≥ 5 peaks. Priority: the two
Enveda timsTOF libraries (same instrument and pipeline as the test set), then everything else.
Caveat: some of these structures may be in MassSpecGym's training set, so absolute fidelity is
optimistic. The **comparisons** (adduct vs adduct, variant vs variant) are what the gate uses.

In [ ]:
t0 = time.time()
cols = ["ingest_lib", "instrument_type", "adduct", "collision_energy_ev", "precursor_mz",
        "normalized_smiles", "inchikey14", "num_peaks"]
t = pq.read_table(TRAIN, columns=cols)
ce_col = t.column("collision_energy_ev").combine_chunks()
if not pa.types.is_list(ce_col.type) and not pa.types.is_large_list(ce_col.type):
    raise TypeError(f"collision_energy_ev is {ce_col.type}, expected a list -- adjust the parser")
offs = ce_col.offsets.to_numpy()
vals = ce_col.values.to_numpy(zero_copy_only=False).astype(np.float64)
n_ce = np.diff(offs)
ce1 = np.full(len(n_ce), np.nan)
one = n_ce == 1
ce1[one] = vals[offs[:-1][one]]

S_all = pd.DataFrame({
    "row": np.arange(t.num_rows),
    "lib": t.column("ingest_lib").to_pandas().astype(object),
    "instr": t.column("instrument_type").to_pandas().astype(object).fillna("").astype(str).str.lower(),
    "adduct": t.column("adduct").to_pandas().astype(object),
    "prec": t.column("precursor_mz").to_numpy(zero_copy_only=False),
    "smiles": t.column("normalized_smiles").to_pandas().astype(object),
    "ik14": t.column("inchikey14").to_pandas().astype(object),
    "npk": t.column("num_peaks").to_numpy(zero_copy_only=False),
    "ce": ce1,
})
del t, ce_col, vals
S_all = S_all[S_all.adduct.isin(TEST_ADDUCTS) & np.isfinite(S_all.ce) & (S_all.ce > 0)
              & (S_all.npk >= 5) & S_all.smiles.notna() & S_all.ik14.notna()]
S_all["tier"] = np.where(S_all.lib.isin(PRIORITY_LIBS), 0, np.where(S_all.instr.str.contains("tims|q-?tof"), 1, 2))
rng = np.random.default_rng(SEED)
S_all["r"] = rng.random(len(S_all))
S = (S_all.sort_values(["tier", "r"]).drop_duplicates(["adduct", "ik14"])
          .groupby("adduct", sort=False).head(N_PER_ADDUCT))
S = S.sort_values(["adduct", "row"]).reset_index(drop=True)

# fetch peaks for the selected rows only
want = dict(zip(S.row, S.index))
MZ, IT = [None] * len(S), [None] * len(S)
base = 0
for batch in pq.ParquetFile(TRAIN).iter_batches(batch_size=200_000, columns=["ms2_mzs", "ms2_normalized_intensities"]):
    hits = [(r - base, want[r]) for r in range(base, base + batch.num_rows) if r in want]
    if hits:
        mzl, itl = batch.column(0), batch.column(1)
        for local, j in hits:
            MZ[j] = np.asarray(mzl[local].as_py(), np.float32)
            IT[j] = np.asarray(itl[local].as_py(), np.float32)
    base += batch.num_rows
# kept as plain lists aligned with S's 0..n-1 index: a DataFrame column of equal-length arrays
# would be coerced to 2-D
assert all(m is not None for m in MZ), "some selected rows were not found in train.parquet"
QS = [prep(m, i, p) for m, i, p in zip(MZ, IT, S.prec)]

comp = S.groupby("adduct").agg(n=("row", "size"), enveda=("tier", lambda x: int((x == 0).sum())),
                               tof=("tier", lambda x: int((x == 1).sum())), ce_median=("ce", "median"))
print(comp.reindex(TEST_ADDUCTS).to_string())
print(f"({time.time()-t0:.0f}s)")
GATE_RESULT["sample"] = comp.reindex(TEST_ADDUCTS).fillna(0).reset_index().to_dict("records")
save_gate()

## Worker call helper

In [ ]:
def run_iceberg(req, ckpt, tag, device="cuda", batch_size=64, sparse_k=100):
    """-> spectra, ok, err, timing, wall_s, above (dict of per-row arrays: what the precursor cut removed)"""
    inp, out = WORK / f"req_{tag}.tsv", WORK / f"resp_{tag}.npz"
    req[["key", "smiles", "adduct", "precursor_mz", "ce", "instrument"]].to_csv(inp, sep="	", index=False)
    cmd = (f'{IPY} {WORKER} --gen "{CK[ckpt]["gen"]}" --inten "{CK[ckpt]["inten"]}" --inp {inp} --out {out} '
           f'--device {device} --batch-size {batch_size} --sparse-k {sparse_k} --prec-margin {PREC_MARGIN}')
    t = time.time()
    sh(cmd, env=VENV_ENV, timeout=6 * 3600)
    z = np.load(out)
    off = z["off"]
    spectra = [(z["mz"][off[i]:off[i + 1]], z["it"][off[i]:off[i + 1]]) for i in range(len(req))]
    above = {k: z[k] for k in ("above_n", "above_share", "above_off")}
    return spectra, z["ok"], z["err"], json.loads(str(z["timing"])), time.time() - t, above

def to_model_ce(ce_ev, prec, mode):
    return ce_ev if mode == "ev" else ce_ev * 500.0 / prec     # ms-pred's own ev_to_nce

def request_from(df, ce_mode, instrument, ce=None):
    ce_ev = df.ce.to_numpy() if ce is None else np.full(len(df), float(ce))
    return pd.DataFrame({"key": df.ik14.to_numpy(), "smiles": df.smiles.to_numpy(), "adduct": df.adduct.to_numpy(),
                         "precursor_mz": df.prec.to_numpy(),
                         "ce": to_model_ce(ce_ev, df.prec.to_numpy(), ce_mode), "instrument": instrument})

## G2 + G3 + G6 — fidelity per adduct, across checkpoint × CE unit × instrument token

For each adduct group (≤ 40 structures): `true` = sim(real, its own prediction); `shuffled` =
sim(real, another structure's prediction); `top1` = the own prediction ranks first among the
group's predictions. Decoys here differ in mass, so top1 is easier than a real ±8.5 ppm window —
read it relative to chance and across variants, not as an MRR estimate.

In [ ]:
def evaluate(S, preds):
    P = [prep(m, i, p) for (m, i), p in zip(preds, S.prec)]
    rows = []
    for ad in TEST_ADDUCTS:
        idx = np.flatnonzero(S.adduct.to_numpy() == ad)
        n = len(idx)
        if n < 5:
            rows.append(dict(adduct=ad, n=n)); continue
        M = np.array([[esim(QS[i], P[j]) for j in idx] for i in idx])
        diag = np.diag(M)
        offd = M[~np.eye(n, dtype=bool)]
        rank = (M > diag[:, None]).sum(1) + 1
        rows.append(dict(adduct=ad, n=n, true=float(np.median(diag)), shuffled=float(np.median(offd)),
                         gap=float(np.median(diag) - np.median(offd)), top1=float((rank == 1).mean()),
                         mrr=float((1 / rank).mean()), chance=1 / n))
    return pd.DataFrame(rows).set_index("adduct")

def sanity(S, preds, ok):
    # revision 2: judged on what the pipeline stores, i.e. AFTER the worker's precursor + PREC_MARGIN cut
    # (the same cut the organisers applied to the test spectra); the m/z test now verifies the cut itself
    good = [bool(o) and len(m) > 0 and np.isfinite(m).all() and np.isfinite(i).all() and m.max() <= p + PREC_MARGIN
            for (m, i), o, p in zip(preds, ok, S.prec)]
    return float(np.mean(good))

GRID = ([(RERUN_CKPT, m, RERUN_INSTR) for m in ("ev", "nce")] if RERUN else
        [(ck, m, i) for ck in CK for m in ("ev", "nce") for i in ("QTOF", "Orbitrap")])
VARIANTS = {}
for ck, ce_mode, instr in GRID:
    tag = f"{ck}_{ce_mode}_{instr}"
    preds, ok, err, timing, wall, above = run_iceberg(request_from(S, ce_mode, instr), ck, tag)
    ev = evaluate(S, preds)
    errs = pd.Series([e.split(":")[0] for e in err if e]).value_counts().to_dict()
    VARIANTS[tag] = dict(table=ev, sanity=sanity(S, preds, ok), errors=errs, timing=timing,
                         ok=ok, err=err, above=above)
    print(f"\n=== {tag}   sanity {VARIANTS[tag]['sanity']:.3f}   errors {errs}   "
          f"{timing['mol_per_s']:.1f} mol/s")
    print(ev.round(3).to_string())

# best variant: judged on [M+H]+ (the adduct MassSpecGym is built around and the most common in test)
score = {k: float(np.nan_to_num(v["table"].get("mrr", pd.Series(dtype=float)).get("[M+H]+", np.nan), nan=-1.0))
         for k, v in VARIANTS.items()}
BEST = max(score, key=score.get)
best_ck = next(c for c in CK if BEST.startswith(c + "_"))       # tags are f"{ckpt}_{ce_mode}_{instrument}"
best_ce, best_instr = BEST[len(best_ck) + 1:].split("_")
tb = VARIANTS[BEST]["table"]
# the first-run rule, kept for comparison only; revision 2 decides coverage with G2b below
covered_g2 = [a for a in TEST_ADDUCTS
              if a in tb.index and "gap" in tb.columns and pd.notna(tb.loc[a, "gap"])
              and tb.loc[a, "gap"] >= COVER_MIN_GAP and tb.loc[a, "top1"] >= COVER_MIN_TOP1_X_CHANCE * tb.loc[a, "chance"]]
print(f"\nBEST variant: {BEST}  (checkpoint {best_ck}, CE as {best_ce}, instrument token {best_instr})")
print(f"G2 rule (mass-disjoint decoys, lenient): {covered_g2}  -- coverage is decided by G2b")

GATE_RESULT["G2"] = dict(best_variant=BEST, checkpoint=best_ck, ce_mode=best_ce, instrument=best_instr,
                         covered_g2_rule=covered_g2,
                         variants={k: dict(sanity=v["sanity"], errors=v["errors"],
                                           table=v["table"].round(4).reset_index().to_dict("records"))
                                   for k, v in VARIANTS.items()})
save_gate()

## G6 — output sanity, per adduct (revision 2)

The first run failed G6 at 0.863 with no worker errors. Hypothesis: `include_unshifted_mz` puts
the unshifted root above the precursor for the three water-loss adducts. The worker now drops
peaks above precursor + 2 Da and records what it dropped. Per adduct: the share of rows that had
drops, the dropped share of intensity, and where the largest dropped peak sat relative to the
precursor (`drop_off`, to compare with the unshifted-root offset `exp_off`). Sanity is judged
after the cut, on what the pipeline actually stores.

In [ ]:
V = VARIANTS[BEST]
A = V["above"]
ad_arr = S.adduct.to_numpy()
err_arr = np.asarray(V["err"]).astype(str)
g6_rows = []
for ad in TEST_ADDUCTS:
    m = ad_arr == ad
    if not m.any():
        continue
    hit = m & (A["above_n"] > 0)
    g6_rows.append(dict(adduct=ad, n=int(m.sum()),
                        rows_with_drop=float(hit.sum() / m.sum()),
                        drop_share=float(np.median(A["above_share"][hit])) if hit.any() else 0.0,
                        drop_off=float(np.nanmedian(A["above_off"][hit])) if hit.any() else np.nan,
                        exp_off=EXPECTED_UNSHIFTED_OFF.get(ad, np.nan),
                        empty_after_cut=float((err_arr[m] == "empty_after_prec_cut").mean()),
                        ok=float(V["ok"][m].mean())))
G6T = pd.DataFrame(g6_rows).set_index("adduct")
print(G6T.round(3).to_string())

drop_ads = [a for a in G6T.index if G6T.loc[a, "rows_with_drop"] > 0]
off_match = all(abs(G6T.loc[a, "drop_off"] - EXPECTED_UNSHIFTED_OFF[a]) < 1.0
                for a in drop_ads if a in EXPECTED_UNSHIFTED_OFF)
hypothesis = bool(drop_ads) and set(drop_ads) <= set(EXPECTED_UNSHIFTED_OFF) and off_match
print(f"\nadducts with dropped peaks: {drop_ads}")
print(f"unshifted-root hypothesis (drops only in water-loss adducts, at the expected offset): "
      f"{'CONFIRMED' if hypothesis else 'NOT confirmed -- read the table'}")
print(f"G6 sanity after the cut: {V['sanity']:.3f} (threshold {SANITY_MIN_OK})")
GATE_RESULT["G6"] = dict(sanity_best=V["sanity"], threshold=SANITY_MIN_OK, passed=V["sanity"] >= SANITY_MIN_OK,
                         judged="after the worker's precursor + 2 Da cut", prec_margin=PREC_MARGIN,
                         unshifted_hypothesis_confirmed=hypothesis, adducts_with_drops=drop_ads,
                         per_adduct=G6T.round(4).reset_index().to_dict("records"))
save_gate()

## G3 — does collision energy actually change the prediction?

In [ ]:
H = S[S.adduct == "[M+H]+"].head(40)
p10, *_ = run_iceberg(request_from(H, best_ce, best_instr, ce=10.0), best_ck, "ce10")
p60, *_ = run_iceberg(request_from(H, best_ce, best_instr, ce=60.0), best_ck, "ce60")
ce_sim = [esim(prep(a[0], a[1], p), prep(b[0], b[1], p)) for a, b, p in zip(p10, p60, H.prec)]
ce_med = float(np.median(ce_sim))
ev_nce = {m: VARIANTS[f"{best_ck}_{m}_{best_instr}"]["table"].loc["[M+H]+", "mrr"] for m in ("ev", "nce")}
print(f"median sim(pred@10 eV, pred@60 eV) = {ce_med:.3f}  -> CE conditions the output: {ce_med < CE_EFFECTIVE_MAX_SIM}")
print(f"[M+H]+ MRR with CE passed as eV {ev_nce['ev']:.3f} | as NCE {ev_nce['nce']:.3f}")
GATE_RESULT["G3"] = dict(median_sim_10_vs_60=ce_med, ce_effective=ce_med < CE_EFFECTIVE_MAX_SIM,
                         mrr_ev=ev_nce["ev"], mrr_nce=ev_nce["nce"], chosen=best_ce)
save_gate()

## G2b — isobaric coverage: rank the truth among its ±8.5 ppm pool window (revision 2)

G2's decoys differ in mass, so a prediction that only lands in the right mass range already wins
(`[M+K]+` reached 32 % top-1 at fidelity 0.028). Tier 2 ranks candidates *inside* a ±8.5 ppm
window, where mass says nothing. Here each sampled structure competes with up to
`ISO_MAX_DECOYS` pool structures within ±8.5 ppm of its exact neutral mass, all simulated with the
query's own adduct, precursor and CE, and scored against the real spectrum. Ties count half
(a truth tied at 0 is not top-1). **An adduct is covered only if** it has ≥ `ISO_MIN_QUERIES`
judged queries, top-1 ≥ `ISO_MIN_LIFT` × chance, and one-sided p < `ISO_MAX_P`. The windows are
centred on the exact mass, not on the measured precursor. That is slightly easier than the real
task, where the ppm error shifts the window.

In [ ]:
import re as _re
# monoisotopic masses as in phase1's formula_mass (label formula -> exact neutral mass, no RDKit needed)
_MONO = {"C": 12.0, "H": 1.0078250319, "N": 14.0030740052, "O": 15.9949146221, "S": 31.97207069,
         "P": 30.97376151, "F": 18.99840320, "Cl": 34.96885271, "Br": 78.9183376, "I": 126.904468,
         "Si": 27.9769265327, "Se": 79.9165196, "B": 11.0093055, "Na": 22.98976928, "K": 38.96370649,
         "Fe": 55.9349375, "D": 2.0141017778, "As": 74.9215965, "Li": 7.0160034, "Sn": 119.9021947}
_TOK = _re.compile(r"([A-Z][a-z]?)(\d*)")
def formula_mass(f):
    if not isinstance(f, str): return np.nan
    s, pos = 0.0, 0
    for m in _TOK.finditer(f):
        if m.start() != pos: return np.nan
        pos = m.end()
        if m.group(1) not in _MONO: return np.nan
        s += _MONO[m.group(1)] * (int(m.group(2)) if m.group(2) else 1)
    return s if pos == len(f) else np.nan

# candidate pool = COCONUT + train structures (as phase1's build_candidate_pool), deduplicated on ik14
cm = pickle.load(open(COCO / "coco_meta.pkl", "rb"))
cmass = np.load(COCO / "coco_mass.npy")
csmi = np.asarray(cm["smiles"], dtype=object)
ckey = np.asarray(cm["keys"], dtype=object)
# decoys are excluded by ik14; if COCONUT's keys were not InChIKeys, a copy of the truth could enter as a decoy
_bad = [k for k in ckey[:: max(1, len(ckey) // 1000)] if not _re.fullmatch(r"[A-Z]{14}", str(k)[:14])]
assert not _bad, f"coco_meta keys are not InChIKey-like (e.g. {_bad[:3]}) -- G2b cannot dedup against the truth"
tr = pq.read_table(TRAIN, columns=["inchikey14", "normalized_smiles", "molecular_formula"]).to_pandas()
tr = tr.dropna(subset=["inchikey14", "normalized_smiles"]).drop_duplicates("inchikey14")
_fm = {f: formula_mass(f) for f in set(tr.molecular_formula.astype(object).tolist())}
tr_mass = np.array([_fm.get(f, np.nan) for f in tr.molecular_formula.astype(object)], dtype=np.float64)
POOL = pd.DataFrame({"k14": np.concatenate([[str(k)[:14] for k in ckey], tr.inchikey14.astype(str).to_numpy()]),
                     "smiles": np.concatenate([csmi, tr.normalized_smiles.astype(object).to_numpy()]),
                     "mass": np.concatenate([cmass.astype(np.float64), tr_mass])})
n_raw = len(POOL)
POOL = (POOL[np.isfinite(POOL.mass) & POOL.smiles.notna()].drop_duplicates("k14")
            .sort_values("mass").reset_index(drop=True))
print(f"pool {n_raw:,} rows -> {len(POOL):,} unique ik14 with a finite mass "
      f"(COCONUT {len(cmass):,}, train {len(tr):,}; COCONUT key sample {ckey[0]!r})")
PM, PK, PS = POOL.mass.to_numpy(), POOL.k14.to_numpy(), POOL.smiles.to_numpy()
k2mass = dict(zip(PK, PM))
del tr, _fm

rng_iso = np.random.default_rng(SEED + 1)
qrows, req_rows = [], []
for qi, r in enumerate(S.itertuples()):
    mt = k2mass.get(r.ik14)
    if mt is None:
        continue
    lo, hi = np.searchsorted(PM, [mt * (1 - ISO_PPM * 1e-6), mt * (1 + ISO_PPM * 1e-6)])
    cand = [j for j in range(lo, hi) if PK[j] != r.ik14]
    win_n = len(cand)
    if len(cand) > ISO_MAX_DECOYS:
        cand = list(rng_iso.choice(cand, ISO_MAX_DECOYS, replace=False))
    qrows.append(dict(qi=qi, adduct=r.adduct, window_n=win_n, n_dec=len(cand), start=len(req_rows)))
    for smi, key in [(r.smiles, r.ik14)] + [(PS[j], PK[j]) for j in cand]:   # truth first
        req_rows.append(dict(key=key, smiles=smi, adduct=r.adduct, precursor_mz=r.prec,
                             ce=to_model_ce(r.ce, r.prec, best_ce), instrument=best_instr))
Q = pd.DataFrame(qrows)
ISO_REQ = pd.DataFrame(req_rows)
print(f"{len(Q)} queries with a pool mass ({len(S) - len(Q)} without), {int((Q.n_dec > 0).sum())} with >= 1 decoy; "
      f"{len(ISO_REQ):,} predictions")
print("full ±8.5 ppm window size per adduct (median):",
      Q.groupby("adduct").window_n.median().reindex(TEST_ADDUCTS).to_dict())

iso_preds, iso_ok, iso_err, iso_timing, iso_wall, _ = run_iceberg(ISO_REQ, best_ck, "iso", batch_size=128)
ISO_P = [prep(m, i, p) for (m, i), p in zip(iso_preds, ISO_REQ.precursor_mz)]

def pb_sf(ps, k):
    """P(X >= k), X = sum of independent Bernoulli(p_i) -- exact Poisson-binomial by DP."""
    d = np.zeros(len(ps) + 1); d[0] = 1.0
    for p in ps:
        d[1:] = d[1:] * (1 - p) + d[:-1] * p
        d[0] *= (1 - p)
    return float(d[k:].sum())

res = []
for q in Q.itertuples():
    sims = np.array([esim(QS[q.qi], ISO_P[q.start + j]) for j in range(q.n_dec + 1)])
    t, dec = sims[0], sims[1:]
    gt, eq = int((dec > t).sum()), int((dec == t).sum())
    res.append(dict(adduct=q.adduct, n_cand=q.n_dec + 1, true=t, rank=1 + gt + eq / 2,
                    top1=gt == 0 and eq == 0, truth_failed=not bool(iso_ok[q.start])))
R = pd.DataFrame(res)
R = R[R.n_cand > 1]
g2b_rows = []
for ad in TEST_ADDUCTS:
    r = R[R.adduct == ad]
    if not len(r):
        g2b_rows.append(dict(adduct=ad, n=0)); continue
    chance_p = 1 / r.n_cand.to_numpy()
    k = int(r.top1.sum())
    lift = r.top1.mean() / chance_p.mean()
    pval = pb_sf(chance_p, k)
    g2b_rows.append(dict(adduct=ad, n=len(r), cands=float(r.n_cand.median()), true=float(r.true.median()),
                         top1=float(r.top1.mean()), chance_top1=float(chance_p.mean()), lift=float(lift),
                         p=pval, mrr=float((1 / r["rank"]).mean()),
                         chance_mrr=float(np.mean([np.mean(1 / np.arange(1, n + 1)) for n in r.n_cand])),
                         truth_failed=int(r.truth_failed.sum()),
                         covered=bool(len(r) >= ISO_MIN_QUERIES and lift >= ISO_MIN_LIFT and pval < ISO_MAX_P)))
G2B = pd.DataFrame(g2b_rows).set_index("adduct")
print(G2B.round(3).to_string())
covered = [a for a in TEST_ADDUCTS if a in G2B.index and G2B.loc[a].get("covered", False) == True]  # noqa: E712 (NaN-safe)
print(f"\nCOVERED (G2b: n >= {ISO_MIN_QUERIES}, top1 >= {ISO_MIN_LIFT}x chance, p < {ISO_MAX_P}): {covered}")
print(f"UNCOVERED -> MetFrag-lite fallback: {[a for a in TEST_ADDUCTS if a not in covered]}")
print(f"G2 rule would have said: {covered_g2}")
GATE_RESULT["G2"].update(covered=covered, uncovered=[a for a in TEST_ADDUCTS if a not in covered])
GATE_RESULT["G2b"] = dict(pool_unique_ik14=len(POOL), ppm=ISO_PPM, max_decoys=ISO_MAX_DECOYS,
                          rule=dict(min_queries=ISO_MIN_QUERIES, min_lift=ISO_MIN_LIFT, max_p=ISO_MAX_P),
                          n_predictions=len(ISO_REQ), mol_per_s=iso_timing["mol_per_s"],
                          window_median=Q.groupby("adduct").window_n.median().to_dict(),
                          table=G2B.round(4).reset_index().to_dict("records"), covered=covered)
save_gate()

## G4 — throughput on real pool structures (COCONUT, 150–1,200 Da)

In re-run mode this is carried from the previous report (it was measured then, and the worker's
precursor cut does not change the forward pass).

In [ ]:
edges = np.linspace(150, 1200, 9)
pick = []
for lo, hi in zip(edges[:-1], edges[1:]):
    ids = np.flatnonzero((cmass >= lo) & (cmass < hi))
    pick += list(rng.choice(ids, size=min(125, len(ids)), replace=False))
pick = np.array(pick)
T = pd.DataFrame({"ik14": ckey[pick], "smiles": csmi[pick], "adduct": "[M+H]+",
                  "prec": cmass[pick] + 1.00727646688, "ce": 30.0})
print(f"{len(T)} COCONUT structures, mass {cmass[pick].min():.0f}-{cmass[pick].max():.0f} Da; "
      f"sample key looks like {ckey[pick[0]]!r} (len {len(str(ckey[pick[0]]))})")

if RERUN:
    G4 = dict(PREV["G4"], carried_from=PREV["started"])
else:
    G4 = {}
    for bs in (64, 128):
        preds, ok, err, timing, wall, _ = run_iceberg(request_from(T, best_ce, best_instr), best_ck, f"tp_gpu{bs}", batch_size=bs)
        G4[f"gpu_bs{bs}"] = dict(mol_per_s=timing["mol_per_s"], wall_s=wall, t_load=timing["t_load"],
                                 ok=float(np.mean(ok)), gpu=timing["gpu"],
                                 errors=pd.Series([e.split(":")[0] for e in err if e]).value_counts().to_dict())
    Tc = T.sample(100, random_state=SEED)
    preds, ok, err, timing, wall, _ = run_iceberg(request_from(Tc, best_ce, best_instr), best_ck, "tp_cpu", device="cpu", batch_size=16)
    G4["cpu"] = dict(mol_per_s=timing["mol_per_s"], wall_s=wall, ok=float(np.mean(ok)), vcpu=os.cpu_count())
best_rate = max(v["mol_per_s"] for k, v in G4.items() if k.startswith("gpu"))
G4["projection_h_per_condition_712k"] = 712_199 / best_rate / 3600
G4["projection_h_per_condition_unique"] = len(POOL) / best_rate / 3600
for k, v in G4.items():
    print(k, v)
print(f"\nprecompute projection: {G4['projection_h_per_condition_712k']:.1f} GPU-h per (adduct, CE) condition "
      f"for the full 712,199-structure pool on this GPU")
GATE_RESULT["G4"] = G4
save_gate()

## G5 (Colab half) — offline wheelhouse + checkpoints → Drive

Builds every wheel the ms-pred environment needs, for this Python (3.12) and CUDA 12.4, plus the
best checkpoint pair and a 20-molecule smoke set with Colab's own predictions as the reference.
Upload `_simlib/gate/kaggle_upload/` as the Kaggle dataset **casmi26-iceberg-offline**, then run
`casmi_simple_sim_gate_kaggle.ipynb` on Kaggle with internet **off**. G5 passes only there.

In [ ]:
if RERUN:
    # the Kaggle half passed in the first run; note that the uploaded worker predates the precursor cut
    GATE_RESULT["G5"] = dict(PREV["G5"], carried_from=PREV["started"],
                             kaggle_half="PASS in the first run (phase0_kaggle_g5.json)",
                             note="kaggle_upload/iceberg_worker.py predates the revision-2 precursor cut; "
                                  "refresh it when the Phase 4 dataset is built")
    print(GATE_RESULT["G5"])
else:
    WH = WORK / "wheelhouse"
    if not any(WH.glob("*.whl")):
        WH.mkdir(parents=True, exist_ok=True)
        sh(f"uv pip install --python {IPY} pip", env=VENV_ENV)
        sh(f"cd {MSPRED_DIR} && uv export --extra cu124 --no-hashes --no-emit-project --no-dev "
           f"--format requirements-txt -o {WORK}/req.txt", env=VENV_ENV)
        sh(f"{IPY} -m pip wheel -q -r {WORK}/req.txt -w {WH} --no-deps "
           f"--extra-index-url https://download.pytorch.org/whl/cu124 "
           f"-f https://data.dgl.ai/wheels/torch-2.6/cu124/repo.html "
           f"-f https://data.pyg.org/whl/torch-2.6.0+cu124.html", env=VENV_ENV, timeout=3 * 3600)
        sh(f"{IPY} -m pip wheel -q {MSPRED_DIR} -w {WH} --no-deps", env=VENV_ENV, timeout=3600)
    whl = sorted(WH.glob("*.whl"))
    n_req = sum(1 for ln in (WORK / "req.txt").read_text().splitlines() if ln.strip() and not ln.startswith(("#", " ", "-")))
    print(f"{len(whl)} wheels for {n_req} requirements (+ ms_pred), {sum(p.stat().st_size for p in whl)/1e9:.2f} GB")

    if UPLOAD.exists():
        shutil.rmtree(UPLOAD)
    (UPLOAD / "wheelhouse").mkdir(parents=True)
    for p in whl:
        shutil.copy2(p, UPLOAD / "wheelhouse" / p.name)
    (UPLOAD / "ckpt").mkdir()
    shutil.copy2(CK[best_ck]["gen"], UPLOAD / "ckpt" / "gen.ckpt")
    shutil.copy2(CK[best_ck]["inten"], UPLOAD / "ckpt" / "inten.ckpt")
    shutil.copy2(WORKER, UPLOAD / "iceberg_worker.py")

    smoke = request_from(S[S.adduct == "[M+H]+"].head(20), best_ce, best_instr)
    ref, *_ = run_iceberg(smoke, best_ck, "smoke")
    smoke.to_csv(UPLOAD / "smoke_requests.tsv", sep="	", index=False)
    lens = np.array([len(m) for m, _ in ref])
    np.savez(UPLOAD / "smoke_reference.npz", off=np.concatenate([[0], np.cumsum(lens)]),
             mz=np.concatenate([m for m, _ in ref]), it=np.concatenate([i for _, i in ref]))
    manifest = dict(checkpoint=best_ck, ce_mode=best_ce, instrument=best_instr,
                    gen_sha1=G1[best_ck]["gen_sha1"], inten_sha1=G1[best_ck]["inten_sha1"],
                    mspred_commit=MSPRED_COMMIT, python=GATE_RESULT["env"]["python"], cuda_extra="cu124",
                    covered_adducts=covered)
    (UPLOAD / "manifest.json").write_text(json.dumps(manifest, indent=1))
    (UPLOAD / "dataset-metadata.json").write_text(json.dumps(
        {"title": "casmi26-iceberg-offline", "id": f"{KAGGLE_USER}/casmi26-iceberg-offline",
         "licenses": [{"name": "other"}]}, indent=1))
    size = sum(p.stat().st_size for p in UPLOAD.rglob("*") if p.is_file())
    print(f"\nwrote {UPLOAD}  ({size/1e9:.2f} GB)")
    GATE_RESULT["G5"] = dict(colab_half="done", upload=str(UPLOAD), n_wheels=len(whl), upload_gb=size / 1e9,
                             kaggle_half="PENDING -- run casmi_simple_sim_gate_kaggle.ipynb")
save_gate()

## Gate report

In [ ]:
def verdict():
    fails = []
    if not GATE_RESULT["G6"]["passed"]:
        fails.append(f"G6 sanity {GATE_RESULT['G6']['sanity_best']:.3f} < {SANITY_MIN_OK}")
    if "[M+H]+" not in GATE_RESULT["G2"]["covered"]:
        fails.append("G2b: [M+H]+ is not covered -- the simulator cannot separate isobaric candidates")
    return fails

fails = verdict()
GATE_RESULT["verdict"] = "FAIL: " + "; ".join(fails) if fails else "PASS (Colab half) -- G5 pending on Kaggle"
GATE_RESULT["finished"] = time.strftime("%Y-%m-%d %H:%M")
save_gate()

print("=" * 88)
print(f"VERDICT: {GATE_RESULT['verdict']}")
print("=" * 88)
print(f"env        {GATE_RESULT['env']}")
print(f"G1         {best_ck}: gen {G1[best_ck]['gen_sha1']}  inten {G1[best_ck]['inten_sha1']}")
print(f"G2         best {BEST}; G2-rule covered {covered_g2}")
print(f"G2b        isobaric-window covered {covered}")
print(f"G3         CE effective {GATE_RESULT['G3']['ce_effective']} (sim10v60 {ce_med:.3f}); "
      f"eV MRR {ev_nce['ev']:.3f} vs NCE {ev_nce['nce']:.3f}")
print(f"G4         GPU {best_rate:.1f} mol/s -> {G4['projection_h_per_condition_712k']:.1f} h per condition; "
      f"CPU {G4['cpu']['mol_per_s']:.2f} mol/s; unique pool {len(POOL):,} -> "
      f"{G4['projection_h_per_condition_unique']:.1f} h per condition")
print(f"G5         {GATE_RESULT['G5']['upload_gb']:.2f} GB upload; Kaggle half: {GATE_RESULT['G5']['kaggle_half']}")
print(f"G6         sanity after cut {GATE_RESULT['G6']['sanity_best']:.3f}; drops in {drop_ads}; "
      f"unshifted-root hypothesis {'confirmed' if hypothesis else 'NOT confirmed'}")
print(f"\nfull report: {GATE / 'gate_report.json'}")
print("\n" + G2B.round(3).to_string())